# MedGemma Service for MedFlow

This notebook runs MedGemma model on Google Colab and exposes it via ngrok for MedFlow to use.

## Instructions:
1. Get your Hugging Face token from: https://huggingface.co/settings/tokens
2. Get your ngrok auth token from: https://dashboard.ngrok.com/get-started/your-authtoken
3. Replace the tokens below
4. Run all cells
5. Copy the ngrok URL and set it as `MEDGEMMA_REMOTE_URL` in MedFlow

In [ ]:
# Install dependencies
!pip install -q transformers torch fastapi uvicorn pyngrok Pillow accelerate

In [ ]:
# Configuration - REPLACE THESE WITH YOUR TOKENS
HF_TOKEN = "hf_YOUR_HUGGINGFACE_TOKEN_HERE"  # Get from https://huggingface.co/settings/tokens
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # Get from https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Imports and setup
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
from pyngrok import ngrok
from huggingface_hub import login
import uvicorn
import nest_asyncio
import torch
import base64
import io
from typing import Optional
import threading

nest_asyncio.apply()

# Login to Hugging Face
login(token=HF_TOKEN)
print("✓ Logged in to Hugging Face")

# Set ngrok auth token
ngrok.set_auth_token(NGROK_TOKEN)
print("✓ ngrok configured")

In [ ]:
# Load MedGemma model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading MedGemma on {device}...")

MODEL_NAME = "google/medgemma-1.5-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print(f"✓ Model loaded on {device.upper()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Create FastAPI app
app = FastAPI(title="MedGemma Inference Service for MedFlow")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Request models
class TextRequest(BaseModel):
    text: str
    max_new_tokens: Optional[int] = 512

class ImageRequest(BaseModel):
    image_base64: str
    prompt: Optional[str] = "Describe this medical image in detail."
    max_new_tokens: Optional[int] = 512

class MultimodalRequest(BaseModel):
    text: str
    image_base64: str
    max_new_tokens: Optional[int] = 512

# Helper functions
def decode_image(base64_string: str) -> Image.Image:
    """Decode base64 image string to PIL Image"""
    image_data = base64.b64decode(base64_string)
    return Image.open(io.BytesIO(image_data)).convert('RGB')

def generate_response(messages, max_new_tokens=512):
    """Generate response from MedGemma model"""
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
        generation = generation[0][input_len:]

    return processor.decode(generation, skip_special_tokens=True)

# API Endpoints
@app.get("/")
async def root():
    return {"status": "MedGemma running", "model": MODEL_NAME, "device": device}

@app.post("/predict_text")
async def predict_text(request: TextRequest):
    """Process text-only medical analysis"""
    try:
        print(f"[TEXT] Input: {request.text[:100]}...")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are a helpful medical AI assistant."}]
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": request.text}]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[TEXT] Output: {response[:100]}...")

        return {
            "input": request.text,
            "response": response,
            "mode": "text-only"
        }

    except Exception as e:
        print(f"[TEXT] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict_image")
async def predict_image(request: ImageRequest):
    """Process medical image analysis"""
    try:
        print(f"[IMAGE] Prompt: {request.prompt[:100]}")
        image = decode_image(request.image_base64)
        print(f"[IMAGE] Image size: {image.size}")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an expert medical imaging assistant."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": request.prompt}
                ]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[IMAGE] Output: {response[:100]}...")

        return {
            "prompt": request.prompt,
            "response": response,
            "mode": "image-only"
        }

    except Exception as e:
        print(f"[IMAGE] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict_multimodal")
async def predict_multimodal(request: MultimodalRequest):
    """Process text + image multimodal analysis"""
    try:
        print(f"[MULTIMODAL] Text: {request.text[:100]}...")
        image = decode_image(request.image_base64)
        print(f"[MULTIMODAL] Image size: {image.size}")

        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an expert medical imaging assistant."}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": request.text}
                ]
            }
        ]

        response = generate_response(messages, request.max_new_tokens)
        print(f"[MULTIMODAL] Output: {response[:100]}...")

        return {
            "input": request.text,
            "response": response,
            "mode": "multimodal"
        }

    except Exception as e:
        print(f"[MULTIMODAL] ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))

print("✓ FastAPI app created")

In [ ]:
# Start the server with ngrok tunnel
print("\n" + "="*60)
print("Starting MedGemma service...")
print("="*60)

# Create ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"{'='*60}")
print(f"\n⚠️  IMPORTANT: Copy the URL above and set it as MEDGEMMA_REMOTE_URL")
print(f"\nExample:")
print(f"  export MEDGEMMA_REMOTE_URL='{public_url}'")
print(f"\nOr add to your .env file:")
print(f"  MEDGEMMA_REMOTE_URL={public_url}")
print(f"\n{'='*60}\n")

# Start uvicorn server in a thread
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run)
thread.start()

print("✓ Server started!")
print("\nThe service is now running. Keep this notebook open.")
print("You can now use MedFlow and it will connect to this Colab instance.")